# CHGNet-PyG Universal Potential

This notebook demonstrates CHGNet with the PyTorch Geometric (PyG) backend. CHGNet is a
charge-informed graph neural network potential that jointly predicts energy, forces, stresses,
and site-wise **magnetic moments**.

We cover:
1. **Static prediction** – energy / forces / stresses / magnetic moments for a single structure
2. **Structure relaxation** – ionic + cell relaxation via the ASE `Relaxer`
3. **Molecular dynamics** – NVT trajectory via the ASE `MolecularDynamics` driver
4. **Training from scratch** – fit a small CHGNet on a custom dataset
5. **Fine-tuning** – continue training from the pre-trained MatPES checkpoint

**Reference:**  
Deng, B. et al. *CHGNet as a pretrained universal neural network potential for charge-informed
atomistic modelling.* Nat. Mach. Intell. (2023) doi:10.1038/s42256-023-00716-3

Author: Bowen Deng

In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import torch
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from pymatgen.core import Lattice, Structure
from pymatgen.io.ase import AseAtomsAdaptor

import matgl
from matgl.ext._ase_pyg import MolecularDynamics, PESCalculator, Relaxer

warnings.filterwarnings("ignore")

## 1. Load the pre-trained CHGNet-PyG model

Two MatPES checkpoints are available:

| Model name | Training functional |
|---|---|
| `BowenD-UCB/CHGNet-PyG-MatPES-r2SCAN-2025.2.10` | r2SCAN |
| `BowenD-UCB/CHGNet-PyG-MatPES-PBE-2025.2.10` | PBE |

Weights are numerically identical to the DGL checkpoints; the only difference is the
message-passing backend. All predictions are in **eV** (energy), **eV/Å** (forces),
**GPa** (stresses), and **μB** (magnetic moments).

In [ ]:
pot = matgl.load_model("BowenD-UCB/CHGNet-PyG-MatPES-r2SCAN-2025.2.10")
pot.eval()
print(pot)

## 2. Static prediction

Use `PESCalculator` (an ASE `Calculator` wrapper) to obtain energy, forces,
stresses, and magnetic moments for any pymatgen `Structure`.

In [ ]:
# BCC iron — a classic magnetic test case
fe_struct = Structure(
    Lattice.cubic(2.87),
    ["Fe", "Fe"],
    [[0, 0, 0], [0.5, 0.5, 0.5]],
)

adaptor = AseAtomsAdaptor()
fe_atoms = adaptor.get_atoms(fe_struct)

calc = PESCalculator(potential=pot)
fe_atoms.set_calculator(calc)

energy = fe_atoms.get_potential_energy()   # eV
forces = fe_atoms.get_forces()              # eV/Å
magmoms = fe_atoms.calc.results["magmoms"] # μB per site

print(f"Energy          : {energy:.4f} eV  ({energy / len(fe_struct):.4f} eV/atom)")
print(f"Max |force|     : {np.abs(forces).max():.4e} eV/Å")
print(f"Magnetic moments: {magmoms.flatten().tolist()}  μB")

### 2a. Direct Potential call (without ASE)

You can also call the `Potential` directly with a PyG graph for finer control
(e.g. batching, or when you need raw tensors for downstream differentiable ops).

In [ ]:
from matgl.ext._pymatgen_pyg import Structure2Graph

conv = Structure2Graph(
    element_types=pot.model.element_types,
    cutoff=pot.model.cutoff,
)

g, lat, state = conv.get_graph(fe_struct)
# Attach Cartesian positions and PBC shift vectors required by Potential
g.pbc_offshift = torch.matmul(g.pbc_offset, lat[0])
g.pos = g.frac_coords @ lat[0]

# out = (energy, forces, stresses, hessian[unused], magmom)
out = pot(g=g, lat=lat, state_attr=state)
energy_t, forces_t, stress_t, _, magmom_t = out

print(f"Energy/atom : {energy_t.item() / len(fe_struct):.4f} eV")
print(f"Forces (eV/Å):\n{forces_t.detach().numpy()}")
print(f"Stress (GPa):\n{stress_t.detach().numpy()}")
print(f"Magmom (μB) : {magmom_t.detach().flatten().tolist()}")

## 3. Structure relaxation

`Relaxer` wraps the ASE cell-filter + FIRE optimizer. By default it relaxes both
ionic positions *and* the cell (`relax_cell=True`).

In [ ]:
# Deliberately distorted CsCl structure
struct = Structure(
    Lattice.cubic(4.5),
    ["Cs", "Cl"],
    [[0, 0, 0], [0.48, 0.52, 0.50]],  # slightly off-centre
)

relaxer = Relaxer(potential=pot)
result = relaxer.relax(struct, fmax=0.01, steps=500)

final_struct = result["final_structure"]
traj = result["trajectory"]

print(f"Initial energy  : {traj.energies[0]:.4f} eV")
print(f"Final energy    : {traj.energies[-1]:.4f} eV")
print(f"Relaxation steps: {len(traj.energies)}")
print(f"Final structure :\n{final_struct}")

In [ ]:
# Trajectory data as a DataFrame
df = traj.as_pandas()
df[["energies", "forces"]].head()

## 4. Molecular dynamics

Run a short NVT simulation (Nosé–Hoover thermostat) on the relaxed structure.
Ensembles available: `"nve"`, `"nvt"`, `"nvt_langevin"`, `"npt"`, `"npt_nose_hoover"`, …

In [ ]:
# Convert the relaxed pymatgen structure to ASE Atoms
atoms = adaptor.get_atoms(final_struct)

# Initialise velocities from a Maxwell–Boltzmann distribution at 300 K
MaxwellBoltzmannDistribution(atoms, temperature_K=300)

driver = MolecularDynamics(
    atoms=atoms,
    potential=pot,
    ensemble="nvt",   # Nosé–Hoover NVT
    temperature=300,   # K
    timestep=2.0,      # fs
    taut=100,          # thermostat time constant (fs)
    logfile="md.log",
    loginterval=10,
)

driver.run(steps=100)
print("MD finished. Final potential energy:", atoms.get_potential_energy(), "eV")

## 5. Training a CHGNet model from scratch

We build a small dataset of Si–O structures (downloaded from the Materials Project)
and train a CHGNet. For a real training run you would include magmom labels and a
much larger dataset — here forces and energies only are used for brevity.

Set `magmom_weight > 0` and include `"magmoms"` in the label dict to train the
magnetic moment head.

In [ ]:
from matgl.ext.pymatgen import get_element_list
from matgl.graph.data import MGLDataset
from matgl.models._chgnet_pyg import CHGNet
from matgl.utils.training import MGLPotentialTrainer, fit_element_refs

In [ ]:
# Obtain your MP API key at https://next-gen.materialsproject.org/api
from mp_api.client import MPRester

mpr = MPRester(api_key="YOUR_API_KEY")
entries = mpr.get_entries_in_chemsys(["Si", "O"])
structures = [e.structure for e in entries]
energies = [e.energy for e in entries]
# Zero forces / stresses for demo; use real DFT values in practice
forces = [np.zeros((len(s), 3)).tolist() for s in structures]
stresses = [np.zeros((3, 3)).tolist() for s in structures]

print(f"{len(structures)} structures downloaded from MP.")

In [ ]:
element_types = get_element_list(structures)

# Fit per-element energy offsets so the model only needs to learn residuals
atomrefs = fit_element_refs(structures, energies, element_types)

# Build PyG graphs with three-body (line-graph) support
converter = Structure2Graph(element_types=element_types, cutoff=6.0)
dataset = MGLDataset(
    structures=structures,
    converter=converter,
    labels={"energies": energies, "forces": forces, "stresses": stresses},
    include_line_graph=True,
    save_cache=False,
)
print(f"Dataset: {len(dataset)} structures, element types: {element_types}")

In [ ]:
# Small CHGNet for demonstration (2 blocks)
model = CHGNet(
    element_types=element_types,
    num_blocks=2,
    dim_atom_embedding=64,
    dim_bond_embedding=64,
    dim_angle_embedding=64,
)

trainer = MGLPotentialTrainer(
    model=model,
    energy_weight=1.0,
    force_weight=1.0,
    stress_weight=0.1,
    magmom_weight=0.0,  # set > 0 when magmom labels are available
    loss="huber_loss",
    lr=1e-3,
    max_epochs=5,        # small for demo; use 100+ for real training
    accelerator="cpu",   # change to "gpu" / "cuda" on a GPU machine
)

potential = trainer.fit(
    dataset=dataset,
    atomrefs=atomrefs,
    save_path="./trained_chgnet_pyg/",
)
print("Training complete.")

In [ ]:
# Load the saved model back
loaded_pot = matgl.load_model("./trained_chgnet_pyg/")
loaded_pot.eval()
print(loaded_pot)

## 6. Fine-tuning the pre-trained CHGNet-PyG

Start from the MatPES r2SCAN checkpoint and continue training on your own dataset.
A lower learning rate (`lr=1e-4`) prevents the pre-trained weights from drifting too fast.

Since the pre-trained CHGNet was trained on the full periodic table, make sure
`element_types` here matches the pre-trained model's `element_types` (or is a subset of it).

In [ ]:
# Load the pre-trained potential
pretrained_pot = matgl.load_model("BowenD-UCB/CHGNet-PyG-MatPES-r2SCAN-2025.2.10")
pretrained_model = pretrained_pot.model

# Per-element energy references from the pre-trained checkpoint
property_offset = pretrained_pot.element_refs.property_offset.numpy()

# Build dataset using the same cutoffs as the pre-trained model
ft_converter = Structure2Graph(
    element_types=pretrained_model.element_types,
    cutoff=pretrained_model.cutoff,
)
ft_dataset = MGLDataset(
    structures=structures,
    converter=ft_converter,
    labels={"energies": energies, "forces": forces, "stresses": stresses},
    include_line_graph=True,
    save_cache=False,
)

ft_trainer = MGLPotentialTrainer(
    model=pretrained_model,
    energy_weight=1.0,
    force_weight=1.0,
    stress_weight=0.1,
    magmom_weight=0.1,  # CHGNet also trains magmom — set 0 if no labels
    lr=1e-4,            # lower LR for fine-tuning
    max_epochs=5,
    accelerator="cpu",
)

ft_potential = ft_trainer.fit(
    dataset=ft_dataset,
    atomrefs=property_offset,
    save_path="./finetuned_chgnet_pyg/",
)
print("Fine-tuning complete.")

## 7. Cleanup

In [ ]:
import os
import shutil

for path in ("md.log", "trained_chgnet_pyg", "finetuned_chgnet_pyg"):
    if os.path.isfile(path):
        os.remove(path)
    elif os.path.isdir(path):
        shutil.rmtree(path)